[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/labs/L2/lab.ipynb)

# Lab 2: columnas que inventamos y clasificación

| | |
|---|---|
| **Parte 1** | Después de T1 y T2, unos 33 minutos: características derivadas, sobreajuste y Ridge |
| **Parte 2** | Después de T3 y T4, unos 28 minutos: la regresión logística y las métricas de clasificación |
| **Parte 3** | Después de T5, T6 y T7, unos 17 minutos: el círculo, tres clases y el perceptrón |
| **En cada tema** | *En vivo*: yo corro y ustedes miran. *Ahora tú*: antes de correr escribes tu predicción en `mi_prediccion` (no se califica), luego corres y comparas. Al final, el **Entregable** del tema |
| **Ritmo** | Lo hacemos juntos, al mismo ritmo: en cada *Ahora tú* todos tenemos los mismos minutos. Lo que falte, en casa |
| **Qué se califica** | Las celdas **Entregable** (tres funciones en Python puro, sin NumPy, y constantes con lo que encontraste) y tu **interpretación** al final. Las constantes se escriben tal cual (por ejemplo `0.4`, no `4/10` ni una variable): el evaluador solo lee valores escritos. Las de la parte 3 valen poco: una prueba cada una |
| **Entrega** | En Moodle, domingo 8 de noviembre a las 23:59. Se acepta hasta el miércoles 11 |

**Antes de empezar**
1. En *Archivo*, *Guardar una copia en Drive*, y trabaja en esa copia.
2. Corre las tres celdas de abajo hasta que aparezca `✅ entorno listo` y el tamaño de los datos.
3. Si algo falla: *Entorno de ejecución*, *Reiniciar y ejecutar todo*.

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dammg_utils import COLORES

from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression, Perceptron
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score, accuracy_score
from sklearn.metrics import log_loss as log_loss_sklearn

du.estilo_dammg()
np.set_printoptions(precision=4, suppress=True)

In [ ]:
df = pd.read_parquet(RUTA_DATOS)
df["est_hora"] = df.station_id.astype(str) + "_" + df.hour_of_day.astype(str)
df["est_hora_finde"] = df.est_hora + "_" + df.is_weekend.astype(str)
df["est_hora_dia"] = df.est_hora + "_" + df.day_of_week.astype(str)
train = df[df.particion == "train"].copy()    # octubre a junio
julio = df[df.particion == "val"].copy()      # para escoger
prueba = df[df.particion == "test"].copy()    # agosto y septiembre: una sola vez, al final
print(df.shape)

## Parte 1: características derivadas, sobreajuste y Ridge
La referencia de hoy es la **tabla por estación y hora** del Lab 1: el promedio de train para cada estación, hora y tipo de día. En julio se equivoca por **1.95** bicis por hora. Todo lo de esta parte se compara contra ese número.

### T1: características derivadas
$\hat y = w_1x_1 + w_2x_2 + \dots + w_nx_n + b$, y cada $x_j$ puede ser una columna que **inventamos** con las que ya hay:

| Columna inventada | En matemáticas | Ejemplo |
|---|---|---|
| rezago | $y^{(t-k)}$ | las salidas de hace una semana |
| categoría | $\mathbb 1[\text{hora}=8]$ | la hora se vuelve 24 columnas de 0 y 1 |
| interacción | $\mathbb 1[\text{est}=51 \wedge \text{hora}=8]$ | `est_hora`: una columna por cada estación y hora |

**Regla de oro:** solo se vale lo que ya se conoce al momento de predecir.

| En Python | Qué hace |
|---|---|
| `OneHotEncoder(handle_unknown="ignore")` | una categoría en columnas de 0 y 1; si en julio aparece una nueva, no truena |
| `ColumnTransformer([(nombre, transformador, columnas)])` | a cada columna, su transformación |
| `serie.shift(k)` | el rezago $k$ (los rezagos ya vienen en los datos) |
| `LinearRegression()` | la recta con todas las columnas |

#### En vivo: sumar columnas contra la interacción
`prueba_columnas` entrena la recta con train y la mide en julio. Primero sumamos la hora y la estación por separado; luego usamos `est_hora_finde`, una columna por cada estación, hora y tipo de día (entre semana o fin de semana).

**¿La recta con la interacción le gana a la tabla (1.95)?**

In [ ]:
CATEGORIAS = ["hour_of_day", "day_of_week", "station_id", "est_hora", "est_hora_finde", "est_hora_dia"]
intentos = {}


def prueba_columnas(columnas):
    cats = [col for col in columnas if col in CATEGORIAS]
    nums = [col for col in columnas if col not in CATEGORIAS]
    partes = ([("cat", OneHotEncoder(handle_unknown="ignore"), cats)] if cats else []) + \
             ([("num", "passthrough", nums)] if nums else [])
    m = make_pipeline(ColumnTransformer(partes), LinearRegression()).fit(train[columnas], train.departures)
    mae_julio = np.mean(np.abs(m.predict(julio[columnas]) - julio.departures))
    intentos[", ".join(columnas)] = round(mae_julio, 3)
    print(f"MAE en julio: {mae_julio:.3f}   con {columnas}")


prueba_columnas(["departures_lag_24h", "departures_lag_168h", "hour_of_day", "station_id"])   # sumadas
prueba_columnas(["departures_lag_24h", "departures_lag_168h", "est_hora_finde"])             # interacción

#### Ahora tú: inventa tu lista
Agrega listas a `mis_listas` y corre. Columnas: `departures_lag_24h`, `departures_lag_168h`, `departures_mean_24h`, `departures_mean_168h`, `hour_of_day`, `day_of_week`, `is_weekend`, `station_id`, `est_hora`, `est_hora_finde`, `est_hora_dia` (estación, hora y día de la semana) y `capacity`.

**¿Qué MAE da `est_hora` sola: más o menos que 1.95?** Después busca la lista con el MAE más bajo. Al final, quítale el `#` a la línea de `arrivals` (las llegadas de esa misma hora) y mira qué pasa.

*Tip: para la fuga, piensa si a las 7:59 ya sabes cuántas bicis llegarán entre las 8 y las 9.*

In [ ]:
# Antes de correr: ¿qué MAE da est_hora sola, más o menos que 1.95?
mi_prediccion = ...    # tu predicción: un número o un texto entre comillas; no se califica

In [ ]:
mis_listas = [
    ["est_hora"],
    # agrega aquí tus listas
    # ["departures_lag_24h", "departures_lag_168h", "est_hora_finde", "arrivals"],   # al final, quítale el #
]
for columnas in mis_listas:
    prueba_columnas(columnas)
pd.Series(intentos, name="MAE en julio").sort_values()

#### Entregable T1
`MEJOR_LISTA`: tu lista con el MAE más bajo en julio **sin fuga**, con las columnas de arriba, entre comillas. El evaluador busca su MAE en una tabla calculada con estos mismos datos y pide 1.900 o menos. `FUGA`: la columna que bajó el MAE pero no se vale.

In [ ]:
# Entregable T1
# Escribe el valor tal cual (por ejemplo 0.4, no 4/10 ni una variable): el evaluador solo lee valores escritos.
MEJOR_LISTA = None    # tu lista con el MAE más bajo, sin fuga; por ejemplo ["departures_lag_168h", "est_hora"]
FUGA = ""             # la columna que no se vale, entre comillas

### T2: sobreajuste, sesgo y varianza, y Ridge
$J(\vec w, b) = \dfrac{1}{2m}\sum_{i=1}^{m}\big(\hat y^{(i)} - y^{(i)}\big)^2 + \dfrac{\lambda}{2m}\sum_{j=1}^{n} w_j^2$

| En matemáticas | En palabras |
|---|---|
| $\frac{1}{2m}\sum(\hat y - y)^2$ | el error de ayer |
| $\sum_j w_j^2$ | la multa: cada peso grande paga; $b$ no paga |
| $\lambda$ | cuánto pesa la multa; con $\lambda = 0$ es la recta de siempre |

Dos restas dan una buena pista: si $E_{train} - E_{ref}$ es grande, es **sesgo** (ni en train le atina); si $E_{julio} - E_{train}$ es grande, es **varianza** (se aprendió train de memoria). Aquí $E_{ref} = 1.95$ es la tabla medida en julio, una referencia aproximada de cuánto se puede bajar.

| En Python | Qué hace |
|---|---|
| `Ridge(alpha=λ)` | la recta con multa; `alpha` es $\lambda$ |
| `LinearRegression()` | lo mismo con $\lambda = 0$ |
| `StandardScaler()` | escalar antes: la multa compara pesos, y los pesos dependen de la unidad |

Con todo train (254 mil filas) Ridge no ayuda: el MAE en julio sube de 1.82 a 2.12 al subir $\lambda$. **Ridge sirve con muchas columnas y pocos datos**, así que aquí entrenamos solo con dos semanas, del 17 al 30 de junio.

#### En vivo: la recta con dos semanas
`prueba_ridge(alpha)` usa `est_hora_dia` (una columna por cada estación, hora y día de la semana: 6,720) y los dos rezagos escalados. Con dos semanas, cada una de esas columnas tiene solo dos filas.

**Con $\lambda = 0$, ¿el MAE de train y el de julio salen parecidos o muy distintos?**

In [ ]:
dos_semanas = train[train.hour >= "2026-06-17"]          # del 17 al 30 de junio
COLS_RIDGE = ["departures_lag_24h", "departures_lag_168h", "est_hora_dia"]
E_REF = 1.95                                              # la tabla del Lab 1, en julio
resultados_ridge = {}


def prueba_ridge(alpha):
    columnas = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), ["est_hora_dia"]),
                                  ("num", StandardScaler(), ["departures_lag_24h", "departures_lag_168h"])])
    recta = LinearRegression() if alpha == 0 else Ridge(alpha=alpha)
    m = make_pipeline(columnas, recta).fit(dos_semanas[COLS_RIDGE], dos_semanas.departures)
    e_train = np.mean(np.abs(m.predict(dos_semanas[COLS_RIDGE]) - dos_semanas.departures))
    e_julio = np.mean(np.abs(m.predict(julio[COLS_RIDGE]) - julio.departures))
    resultados_ridge[alpha] = {"train": e_train, "julio": e_julio}
    print(f"λ = {alpha:<6} MAE en train: {e_train:.2f}   en julio: {e_julio:.2f}"
          f"   train − ref = {e_train - E_REF:+.2f}   julio − train = {e_julio - e_train:+.2f}")


print(f"{len(dos_semanas):,} filas para entrenar")
prueba_ridge(0)

#### Ahora tú: barre λ
Completa la rejilla $\lambda$ = 0, 0.1, 1, 3, 10, 100, 1000 y corre. La gráfica pone train y julio contra $\lambda$, con la tabla (1.95) de referencia.

**¿Qué $\lambda$ da el MAE más bajo en julio?** Y con $\lambda = 1000$, ¿qué pasa con el MAE de train?

*Tip: escoge $\lambda$ con julio, nunca con train: en train siempre gana $\lambda = 0$.*

In [ ]:
# Antes de correr: ¿qué λ de la rejilla da el MAE más bajo en julio?
mi_prediccion = ...    # tu predicción: un número o un texto entre comillas; no se califica

In [ ]:
for alpha in [0, 1000]:          # completa la rejilla: 0, 0.1, 1, 3, 10, 100, 1000
    prueba_ridge(alpha)
tabla_ridge = pd.DataFrame(resultados_ridge).T.sort_index()
fig, ax = plt.subplots(figsize=(7, 4))
posiciones = range(len(tabla_ridge))
ax.plot(posiciones, tabla_ridge.train, marker="o", color=COLORES["w"], label="train (dos semanas)")
ax.plot(posiciones, tabla_ridge.julio, marker="o", color=COLORES["lambda"], label="julio")
ax.axhline(E_REF, color=COLORES["tinta"], lw=1, ls=":", label="la tabla del Lab 1 (1.95)")
ax.set_xticks(list(posiciones), [str(a) for a in tabla_ridge.index])
ax.set(xlabel="λ", ylabel="MAE (bicis por hora)")
ax.legend()
plt.show()
tabla_ridge.round(2)

#### Entregable T2
`ALPHA`: el $\lambda$ de la rejilla con el MAE más bajo en julio. `DIAG_ALPHA_0` y `DIAG_ALPHA_1000`: `"sesgo"` o `"varianza"`, con $E_{ref} = 1.95$.

In [ ]:
# Entregable T2
# Escribe el valor tal cual (por ejemplo 0.4, no 4/10 ni una variable): el evaluador solo lee valores escritos.
ALPHA = None              # el λ de la rejilla con el MAE más bajo en julio
DIAG_ALPHA_0 = ""         # "sesgo" o "varianza": el diagnóstico con λ = 0
DIAG_ALPHA_1000 = ""      # "sesgo" o "varianza": el diagnóstico con λ = 1000

### Tus preguntas (parte 1)
Anota una duda para la siguiente sesión. No se califica.

_Escribe aquí._

## Parte 2: la regresión logística y las métricas
Cambia la pregunta: ya no es cuántas bicis salen, sino **¿habrá un desbalance fuerte?** Sí o no. Cada alerta es un viaje del camión que reacomoda bicis.

In [ ]:
# @title Si Colab se reinició, corre esto (después de las dos celdas de preparación del inicio)
if 'julio' not in globals():
    df = pd.read_parquet(RUTA_DATOS)
    df["est_hora"] = df.station_id.astype(str) + "_" + df.hour_of_day.astype(str)
    df["est_hora_finde"] = df.est_hora + "_" + df.is_weekend.astype(str)
    df["est_hora_dia"] = df.est_hora + "_" + df.day_of_week.astype(str)
    train = df[df.particion == "train"].copy()    # octubre a junio
    julio = df[df.particion == "val"].copy()      # para escoger
    prueba = df[df.particion == "test"].copy()    # agosto y septiembre: una sola vez, al final
    print('Datos recargados.')
else:
    print('Los datos siguen cargados: no hace falta.')

### T3: regresión logística
| En matemáticas | En palabras |
|---|---|
| $z = \vec w \cdot \vec x + b$ | la recta de siempre, que da cualquier número |
| $\hat p = \sigma(z) = \dfrac{1}{1+e^{-z}}$ | la sigmoide la aplasta entre 0 y 1: una probabilidad |
| alerta si $\hat p \ge t$ | el umbral $t$ lo escogemos nosotros |
| $-\frac{1}{m}\sum \big[y\log\hat p + (1-y)\log(1-\hat p)\big]$ | la log-loss: castiga equivocarse con seguridad |

| En Python | Qué hace |
|---|---|
| `LogisticRegression(C=1.0, max_iter=1000)` | la logística; `C` es $1/\lambda$: más `C`, menos multa |
| `.predict_proba(X)[:, 1]` | $\hat p$ de cada fila |
| `.predict(X)` | alerta si $\hat p \ge 0.5$ |

#### En vivo: la etiqueta
Un desbalance fuerte es $|$flujo neto de esa hora y las dos siguientes$| \ge u$. El corte $u$ es el percentil 85 de train y se congela. Decidimos casi un día antes: lo más reciente que usamos es la misma hora de ayer.

In [ ]:
u = float(train.future_net.abs().quantile(0.85))     # el percentil 85 de train: 7 bicis; se congela
for d in (train, julio, prueba):
    d["y"] = (d.future_net.abs() >= u).astype(int)
print(f"corte u = {u}   desbalances: train {train.y.mean():.1%}   julio {julio.y.mean():.1%}")

#### En vivo: una variable, de la recta a la curva
$x$ = el desbalance de hace una semana, sin signo (`|net_lag_168h|`). Los puntos son la proporción real de desbalances para cada $x$. Primero una recta, luego la logística.

**Con $x = 60$, ¿qué "probabilidad" da la recta?**

In [ ]:
x_train = pd.DataFrame({"x": train.net_lag_168h.abs()})
recta01 = LinearRegression().fit(x_train, train.y)
m1 = LogisticRegression().fit(x_train, train.y)
w1, b1 = m1.coef_[0][0], m1.intercept_[0]
print(f"la recta con x = 60: {recta01.predict(pd.DataFrame({'x': [60]}))[0]:.2f}")
print(f"la logística: w = {w1:.2f}, b = {b1:.2f}; p̂ = 0.5 cuando x = {-b1 / w1:.1f}")

proporcion = train.y.groupby(x_train.x.round()).mean().loc[:30]
xs = np.linspace(0, 30, 200)
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(proporcion.index, proporcion.values, color=COLORES["datos"], zorder=3, label="proporción real")
ax.plot(xs, recta01.predict(pd.DataFrame({"x": xs})), color=COLORES["error"], label="recta")
ax.plot(xs, 1 / (1 + np.exp(-(w1 * xs + b1))), color=COLORES["yhat"], label="logística")
ax.set(xlabel="|desbalance| hace una semana", ylabel="p̂ de desbalance")
ax.legend()
plt.show()

#### En vivo: la logística con 12 columnas
Rezagos del flujo neto con y sin signo, salidas, capacidad, fin de semana y hora pico. El escalador se ajusta solo con train, dentro del pipeline. Abajo, tres filas de julio: la de $\hat p$ más bajo, la de en medio y la más alta.

**¿Cuánto baja la log-loss en julio contra decir 0.5 a todo y contra decir la tasa de train a todo?**

In [ ]:
HORAS_PICO = [7, 8, 9, 17, 18, 19, 20]


def variables(d):
    """Decidimos casi un día antes: lo más reciente que usamos es la misma hora de ayer. Rezagos, promedios, calendario y capacidad."""
    return pd.DataFrame({
        "net_lag_24h": d.net_lag_24h, "net_lag_168h": d.net_lag_168h, "net_mean_24h": d.net_mean_24h,
        "abs_net_lag_24h": d.net_lag_24h.abs(), "abs_net_lag_168h": d.net_lag_168h.abs(),
        "abs_net_mean_24h": d.net_mean_24h.abs(),
        "departures_mean_24h": d.departures_mean_24h, "departures_mean_168h": d.departures_mean_168h,
        "departures_lag_168h": d.departures_lag_168h, "capacity": d.capacity,
        "is_weekend": d.is_weekend, "pico": d.hour_of_day.isin(HORAS_PICO).astype(int),
    }, index=d.index)


X_train, X_julio, X_prueba = variables(train), variables(julio), variables(prueba)
modelo = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000)).fit(X_train, train.y)
p_julio = modelo.predict_proba(X_julio)[:, 1]
orden = np.argsort(p_julio)
filas = [orden[0], orden[len(orden) // 2], orden[-1]]
print(julio.iloc[filas][["station_id", "hour", "net_lag_168h", "y"]].assign(p_hat=p_julio[filas].round(2)))
print(f"\nlog-loss en julio:   0.5 a todo = {log_loss_sklearn(julio.y, np.full(len(julio), 0.5)):.3f}"
      f"   tasa de train a todo = {log_loss_sklearn(julio.y, np.full(len(julio), train.y.mean())):.3f}"
      f"   logística = {log_loss_sklearn(julio.y, p_julio):.3f}")

#### Ahora tú: la sigmoide y la log-loss a mano
Escribe `sigmoide` y `log_loss` en la celda del Entregable, en Python puro: `for`, `zip` y `math`, sin NumPy. La celda de después las compara con scikit-learn.

**¿Cuánto vale $\hat p$ con $x = 10$, con la logística de una variable?**

*Tip: $z = w \cdot 10 + b$, con los $w$ y $b$ que salieron arriba.*

In [ ]:
# Antes de correr: ¿cuánto vale p̂ con x = 10?
mi_prediccion = ...    # tu predicción: un número o un texto entre comillas; no se califica

#### Entregable T3: `sigmoide` y `log_loss`
Se califican con pruebas aparte: no cambies nombres ni argumentos.

In [ ]:
import math


def sigmoide(z):
    """σ(z) = 1 / (1 + e^(−z)) para un número z. Regresa un float. Con z = ±1000 no debe tronar."""
    # TODO: 1 / (1 + math.exp(-z)). Pista: con z = -1000, math.exp(1000) truena; si z < 0, usa e^z / (1 + e^z).
    raise NotImplementedError("Escribe sigmoide.")


def log_loss(y, p):
    """Log-loss promedio: −(1/m) Σ [y log p + (1 − y) log(1 − p)].

    y : lista de etiquetas 0 o 1.   p : lista de probabilidades p̂ de la clase 1, del mismo largo.
    Recorta cada p a [1e-12, 1 − 1e-12] antes del log, para no calcular log(0). Regresa un float.
    """
    # TODO: un for sobre zip(y, p): recorta pi, suma y*log(p) + (1 − y)*log(1 − p), y al final −total / m.
    raise NotImplementedError("Escribe log_loss.")

In [ ]:
try:
    p10 = m1.predict_proba(pd.DataFrame({"x": [10]}))[0, 1]
    print(f"p̂ con x = 10:   tu sigmoide = {sigmoide(w1 * 10 + b1):.3f}   scikit-learn = {p10:.3f}")
    print(f"sigmoide(-1000) = {sigmoide(-1000)}   (sin error)")
    print(f"log-loss en julio:   tu log_loss = {log_loss(julio.y.tolist(), p_julio.tolist()):.4f}"
          f"   scikit-learn = {log_loss_sklearn(julio.y, p_julio):.4f}")
except NotImplementedError as e:
    print('Todavía falta:', e)

### T4: métricas de clasificación
Con $\hat p$ y un umbral $t$, cada fila cae en una de cuatro casillas:

| | dijo calma (0) | dijo alerta (1) |
|---|---|---|
| **pasó calma (0)** | TN: calma bien leída | FP: viaje en balde |
| **pasó desbalance (1)** | FN: desbalance perdido | TP: alerta cierta |

| Métrica | Fórmula | En palabras |
|---|---|---|
| exactitud | $\frac{TN + TP}{m}$ | cuántas acertó, de todas |
| precision | $\frac{TP}{TP + FP}$ | de las alertas, cuántas eran ciertas |
| recall | $\frac{TP}{TP + FN}$ | de los desbalances, cuántos atrapó |

| En Python | Qué hace |
|---|---|
| `confusion_matrix(y, y_pred).ravel()` | (TN, FP, FN, TP): filas = lo que pasó, primero el 0 |
| `precision_score`, `recall_score`, `accuracy_score` | las tres métricas; `zero_division=0` si no hay alertas |

#### En vivo: la matriz con t = 0.5
Comparamos la logística con dos referencias: decir siempre "no hay desbalance" y la regla "alerta si hace una semana hubo 7 o más".

**Antes de correr: ¿qué exactitud tiene decir siempre que no?**

In [ ]:
def resumen(nombre, alerta):
    tn, fp, fn, tp = confusion_matrix(julio.y, alerta).ravel()
    print(f"{nombre:20s} TN = {tn:6,}  FP = {fp:5,}  FN = {fn:5,}  TP = {tp:5,}   alertas = {alerta.mean():5.1%}"
          f"   exactitud = {accuracy_score(julio.y, alerta):.1%}   precision = {precision_score(julio.y, alerta, zero_division=0):.2f}"
          f"   recall = {recall_score(julio.y, alerta):.2f}")


alerta = (p_julio >= 0.5).astype(int)
resumen("nunca", np.zeros(len(julio), dtype=int))
resumen("regla: 7 o más", (julio.net_lag_168h.abs() >= 7).astype(int).to_numpy())
resumen("logística, t = 0.5", alerta)

#### Ahora tú: la matriz a mano
Escribe `matriz_confusion` en la celda del Entregable, en Python puro. La celda de después la compara con scikit-learn con $t = 0.5$ y con $t = 0.3$.

**Si bajas el umbral de 0.5 a 0.3, ¿los falsos positivos suben o bajan? ¿Cuántas veces?**

*Tip: con un umbral más bajo, más filas pasan a "alerta".*

In [ ]:
# Antes de correr: con t = 0.3, ¿los FP suben o bajan, y cuántas veces?
mi_prediccion = ...    # tu predicción: un número o un texto entre comillas; no se califica

#### Entregable T4: `matriz_confusion`

In [ ]:
def matriz_confusion(y, y_pred):
    """Cuenta los cuatro casos y regresa la tupla (TN, FP, FN, TP) como ints.

    Es el mismo orden que confusion_matrix(y, y_pred).ravel() de scikit-learn: primero el 0.
    y, y_pred : listas de etiquetas 0 o 1, del mismo largo.
    """
    # TODO: un for sobre zip(y, y_pred) que cuente los cuatro casos; regresa (tn, fp, fn, tp).
    raise NotImplementedError("Escribe matriz_confusion.")

In [ ]:
try:
    for t in (0.5, 0.3):
        al = (p_julio >= t).astype(int)
        print(f"t = {t}:   tu matriz = {matriz_confusion(julio.y.tolist(), al.tolist())}"
              f"   scikit-learn = {tuple(int(v) for v in confusion_matrix(julio.y, al).ravel())}")
except NotImplementedError as e:
    print('Todavía falta:', e)

#### Ahora tú: el umbral con presupuesto
El presupuesto es el 25 % de las estaciones y horas de julio: no hay camión para más. Mueve el umbral $t$ y mira alertas, precision y recall. Busca el **menor** $t$ que cabe en el 25 %.

**¿El umbral que escojas será mayor o menor que 0.5?**

*Tip: bajar el umbral manda más alertas: atrapas más desbalances, pero más viajes son en balde.*

In [ ]:
# Antes de correr: ¿el umbral será mayor o menor que 0.5?
mi_prediccion = ...    # tu predicción: un número o un texto entre comillas; no se califica

In [ ]:
PRESUPUESTO = 0.25


def ver_umbral(t):
    al = (p_julio >= t).astype(int)
    print(f"t = {t:.2f}   alertas = {al.mean():6.1%}   precision = {precision_score(julio.y, al, zero_division=0):.2f}"
          f"   recall = {recall_score(julio.y, al):.2f}   {'cabe' if al.mean() <= PRESUPUESTO else 'se pasa'} en el 25 %")


try:
    import ipywidgets as widgets
    widgets.interact(ver_umbral, t=widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.01, description="umbral t"))
except ImportError:
    print("No hay ipywidgets: usa la celda de abajo.")

In [ ]:
# Si el widget no aparece: cambia t y vuelve a correr
ver_umbral(0.5)

#### Entregable T4: tu umbral
`UMBRAL_ALERTA`: el menor umbral de la rejilla 0.00, 0.01, ..., 1.00 cuyas alertas en julio caben en el 25 %.

In [ ]:
# Entregable T4: tu umbral
# Escribe el valor tal cual (por ejemplo 0.4, no 4/10 ni una variable): el evaluador solo lee valores escritos.
UMBRAL_ALERTA = None      # el menor umbral de la rejilla 0.00, 0.01, ..., 1.00 que cabe en el 25 %

#### Una sola vez: agosto y septiembre
La celda de abajo revisa tu umbral contra la rejilla completa y luego lo prueba **una sola vez** en agosto y septiembre. Después de verlo no se cambia ninguna decisión.

In [ ]:
if UMBRAL_ALERTA is None:
    print("Primero escribe tu UMBRAL_ALERTA en el entregable de arriba; esta celda no abre agosto y septiembre sin él.")
else:
    rejilla = np.round(np.arange(0, 1.001, 0.01), 2)
    alertas_rejilla = np.array([(p_julio >= a).mean() for a in rejilla])
    umbral_rejilla = float(rejilla[np.argmax(alertas_rejilla <= PRESUPUESTO)])
    print(f"el menor de la rejilla que cabe: {umbral_rejilla}   el tuyo: {UMBRAL_ALERTA}")
    ver_umbral(UMBRAL_ALERTA)

    alerta_prueba = (modelo.predict_proba(X_prueba)[:, 1] >= UMBRAL_ALERTA).astype(int)
    print(f"\nprueba (ago y sep), una sola vez: alertas = {alerta_prueba.mean():.1%}"
          f"   precision = {precision_score(prueba.y, alerta_prueba):.2f}   recall = {recall_score(prueba.y, alerta_prueba):.2f}")
    print(f"¿se cumplió el presupuesto? {'sí' if alerta_prueba.mean() <= PRESUPUESTO else 'no'}")
    print(f"desbalances: julio {julio.y.mean():.1%}   prueba {prueba.y.mean():.1%}")

### Tus preguntas (parte 2)
Anota una duda para la siguiente sesión. No se califica.

_Escribe aquí._

## Parte 3: más allá de una recta
Tres temas cortos: una columna inventada que dobla la frontera, tres clases en lugar de dos y el perceptrón, la neurona del Módulo VIII.

In [ ]:
# @title Si Colab se reinició, corre esto (después de las dos celdas de preparación del inicio)
if 'modelo' not in globals():
    df = pd.read_parquet(RUTA_DATOS)
    df["est_hora"] = df.station_id.astype(str) + "_" + df.hour_of_day.astype(str)
    df["est_hora_finde"] = df.est_hora + "_" + df.is_weekend.astype(str)
    df["est_hora_dia"] = df.est_hora + "_" + df.day_of_week.astype(str)
    train = df[df.particion == "train"].copy()    # octubre a junio
    julio = df[df.particion == "val"].copy()      # para escoger
    prueba = df[df.particion == "test"].copy()    # agosto y septiembre: una sola vez, al final

    u = float(train.future_net.abs().quantile(0.85))     # el percentil 85 de train: 7 bicis; se congela
    for d in (train, julio, prueba):
        d["y"] = (d.future_net.abs() >= u).astype(int)

    HORAS_PICO = [7, 8, 9, 17, 18, 19, 20]


    def variables(d):
        """Decidimos casi un día antes: lo más reciente que usamos es la misma hora de ayer. Rezagos, promedios, calendario y capacidad."""
        return pd.DataFrame({
            "net_lag_24h": d.net_lag_24h, "net_lag_168h": d.net_lag_168h, "net_mean_24h": d.net_mean_24h,
            "abs_net_lag_24h": d.net_lag_24h.abs(), "abs_net_lag_168h": d.net_lag_168h.abs(),
            "abs_net_mean_24h": d.net_mean_24h.abs(),
            "departures_mean_24h": d.departures_mean_24h, "departures_mean_168h": d.departures_mean_168h,
            "departures_lag_168h": d.departures_lag_168h, "capacity": d.capacity,
            "is_weekend": d.is_weekend, "pico": d.hour_of_day.isin(HORAS_PICO).astype(int),
        }, index=d.index)


    X_train, X_julio, X_prueba = variables(train), variables(julio), variables(prueba)
    modelo = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000)).fit(X_train, train.y)
    p_julio = modelo.predict_proba(X_julio)[:, 1]
    print('Datos y logística recargados.')
else:
    print('Todo sigue cargado: no hace falta.')

### T5: transformaciones en el espacio de variables
$z = w_1x_1 + w_2x_2 + w_3\big(x_1^2 + x_2^2\big) + b$: la frontera $z = 0$ ya no es una recta, es un círculo. Es T1 otra vez: una columna inventada.

| En Python | Qué hace |
|---|---|
| `PolynomialFeatures(degree=2)` | crea $x_1, x_2, x_1^2, x_1x_2, x_2^2$ |
| `make_pipeline(PolynomialFeatures(2), StandardScaler(), LogisticRegression())` | columnas nuevas, escalar y la logística |

#### En vivo: el anillo con una recta
Dos clases en anillo, 300 puntos. `prueba_grado(g)` ajusta la logística con columnas de grado `g` y dibuja la frontera.

**¿Qué recta separa el anillo?**

In [ ]:
from sklearn.datasets import make_circles

Xc, yc = make_circles(n_samples=300, noise=0.08, factor=0.45, random_state=0)


def prueba_grado(g):
    m = make_pipeline(PolynomialFeatures(g), StandardScaler(), LogisticRegression()).fit(Xc, yc)
    print(f"grado {g}: exactitud = {m.score(Xc, yc):.0%}")
    malla = np.linspace(-1.4, 1.4, 200)
    g1, g2 = np.meshgrid(malla, malla)
    zona = m.predict(np.c_[g1.ravel(), g2.ravel()]).reshape(g1.shape)
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.contourf(g1, g2, zona, alpha=0.15, cmap="coolwarm")
    ax.scatter(Xc[:, 0], Xc[:, 1], c=yc, cmap="coolwarm", s=12)
    ax.set(title=f"grado {g}", xlabel="x1", ylabel="x2")
    plt.show()


prueba_grado(1)

#### Ahora tú: sube el grado
Cambia el grado a 2 y a 3.

**¿Qué grado basta para separar el anillo?**

In [ ]:
# Antes de correr: ¿qué grado basta para separar el anillo?
mi_prediccion = ...    # tu predicción: un número o un texto entre comillas; no se califica

In [ ]:
prueba_grado(1)      # cambia a 2 y luego a 3

#### Entregable T5
`GRADO_MINIMO`: el grado más chico con el que la exactitud llega a 100 %.

In [ ]:
# Entregable T5
# Escribe el valor tal cual (por ejemplo 0.4, no 4/10 ni una variable): el evaluador solo lee valores escritos.
GRADO_MINIMO = None       # el grado más chico que separa el anillo

### T6: multiclase
Tres clases de salidas: pocas (0 o 1), normales (2 a 5) y muchas (6 o más).

| Forma | En matemáticas | En palabras |
|---|---|---|
| uno contra el resto | 3 logísticas, gana la de $\hat p_k$ más alta | una pregunta de sí o no por clase |
| softmax | $\hat p_k = \dfrac{e^{z_k}}{\sum_j e^{z_j}}$ | un $z$ por clase; las tres $\hat p$ suman 1 |

| En Python | Qué hace |
|---|---|
| `LogisticRegression()` | con 3 clases hace softmax solo |
| `OneVsRestClassifier(LogisticRegression())` | uno contra el resto |
| `confusion_matrix(y, y_pred, labels=[...])` | la matriz de 3 por 3, en el orden que tú fijas |

#### En vivo: tres clases con softmax
`prueba_multiclase` entrena con train y mide en julio. La matriz se lee por filas: lo que pasó contra lo que dijo el modelo.

**¿Qué par de clases confunde más?**

In [ ]:
q1, q2 = train.departures.quantile([1 / 3, 2 / 3])
clase = lambda v: np.where(v <= q1, "pocas", np.where(v <= q2, "normales", "muchas"))
COLS3 = ["departures_lag_24h", "departures_lag_168h", "departures_mean_168h", "hour_of_day", "is_weekend"]
ETIQUETAS = ["pocas", "normales", "muchas"]


def prueba_multiclase(estimador):
    m = make_pipeline(StandardScaler(), estimador).fit(train[COLS3], clase(train.departures))
    cm = confusion_matrix(clase(julio.departures), m.predict(julio[COLS3]), labels=ETIQUETAS)
    print(f"exactitud en julio: {np.trace(cm) / cm.sum():.1%}")
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.imshow(cm, cmap="Blues")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks(range(3), ETIQUETAS)
    ax.set_yticks(range(3), ETIQUETAS)
    ax.set(xlabel="lo que dijo el modelo", ylabel="lo que pasó")
    ax.grid(False)
    plt.show()
    return m


m3 = prueba_multiclase(LogisticRegression(max_iter=1000))
fila = julio[julio.hour_of_day == 2][COLS3].iloc[[0]]
p_fila = dict(zip(m3.classes_, m3.predict_proba(fila)[0]))
print("una fila de julio a las 2 a.m.:", "   ".join(f"{k} {p_fila[k]:.2f}" for k in ETIQUETAS))

#### Ahora tú: uno contra el resto
Cambia el estimador por `OneVsRestClassifier(LogisticRegression(max_iter=1000))`.

**¿Uno contra el resto acierta más o menos que softmax?**

In [ ]:
# Antes de correr: ¿uno contra el resto acierta más o menos que softmax?
mi_prediccion = ...    # tu predicción: un número o un texto entre comillas; no se califica

In [ ]:
prueba_multiclase(LogisticRegression(max_iter=1000));      # cámbialo por uno contra el resto

#### Entregable T6
`MAS_CONFUNDE`: en la matriz de **softmax** (la del En vivo), la casilla más grande fuera de la diagonal, como `(lo que pasó, lo que dijo el modelo)`. Con las etiquetas `"pocas"`, `"normales"` o `"muchas"`.

In [ ]:
# Entregable T6
# Escribe el valor tal cual (por ejemplo 0.4, no 4/10 ni una variable): el evaluador solo lee valores escritos.
MAS_CONFUNDE = None       # (lo que pasó, lo que dijo el modelo), por ejemplo ("pocas", "muchas")

### T7: logística y perceptrón
| En matemáticas | En palabras |
|---|---|
| $\hat y = \mathbb 1[\vec w \cdot \vec x + b \ge 0]$ | un escalón: dice 0 o 1, nunca una probabilidad |
| $\vec w \leftarrow \vec w + \eta\,(y - \hat y)\,\vec x$ | si acierta no se mueve; si falla, empuja $\vec w$ hacia el punto |

La regla "7 o más" era un escalón a mano; el perceptrón aprende el escalón y la logística lo suaviza. Es la neurona del Módulo VIII.

| En Python | Qué hace |
|---|---|
| `Perceptron(eta0=η, max_iter=50, random_state=0)` | `eta0` es $\eta$, el tamaño del paso; `max_iter`, las vueltas a los datos |

#### En vivo: perceptrón contra logística
Mismas 12 columnas y misma etiqueta que en la parte 2.

**¿Cuál atrapa más desbalances?**

In [ ]:
umbral_t4 = globals().get("UMBRAL_ALERTA")     # tu umbral de T4


def prueba_perceptron(eta0):
    perc = make_pipeline(StandardScaler(), Perceptron(eta0=eta0, max_iter=50, random_state=0)).fit(X_train, train.y)
    pred = perc.predict(X_julio)
    print(f"perceptrón, eta0 = {eta0:<4}  alertas = {pred.mean():5.1%}   precision = {precision_score(julio.y, pred, zero_division=0):.2f}"
          f"   recall = {recall_score(julio.y, pred):.2f}")


for t in (0.5,) if umbral_t4 is None else (0.5, umbral_t4):
    al = (p_julio >= t).astype(int)
    print(f"logística, t = {t:<5}    alertas = {al.mean():5.1%}   precision = {precision_score(julio.y, al):.2f}"
          f"   recall = {recall_score(julio.y, al):.2f}")
prueba_perceptron(0.1)

#### Ahora tú: cambia η
Cambia `eta0` de 0.1 a 1.0.

**¿Cambia algo?**

*Tip: el perceptrón arranca con $\vec w = 0$ y $b = 0$.*

In [ ]:
# Antes de correr: con eta0 = 1.0, ¿cambia algo?
mi_prediccion = ...    # tu predicción: un número o un texto entre comillas; no se califica

In [ ]:
prueba_perceptron(0.1)      # cámbialo a 1.0

#### Entregable T7
`CON_UMBRAL`: `"logistica"` o `"perceptron"`, sin acento: ¿cuál te da una probabilidad $\hat p$ (`predict_proba`) para aplicar el umbral con presupuesto de T4?

In [ ]:
# Entregable T7
# Escribe el valor tal cual (por ejemplo 0.4, no 4/10 ni una variable): el evaluador solo lee valores escritos.
CON_UMBRAL = ""           # "logistica" o "perceptron": ¿cuál te da una probabilidad p̂ (predict_proba)?

### Tus preguntas (parte 3)
Anota una duda para la siguiente sesión. No se califica.

_Escribe aquí._

## Tu interpretación: el umbral en prueba
¿Qué `UMBRAL_ALERTA` escogiste, cuántas alertas da en julio y con qué precision y recall? En prueba, ¿se cumplió el presupuesto? Di qué observaste y, aparte, qué podría explicarlo. De 3 a 5 líneas, con tus números.

**Tu respuesta:**

**Para entregar:** *Entorno de ejecución*, *Reiniciar y ejecutar todo*. Luego *Archivo*, *Descargar .ipynb*, y súbelo a Moodle. No cambies los nombres de las funciones ni de las constantes de los Entregables.